# Planetary Analogs — בדיקת דאטה לפני מידול
הריצו את כל התאים לפי הסדר. ברירת המחדל מורידה את המאגר המורחב מ־GitHub. אפשר להעלות ל־Colab את העותק המקומי דרך חלונית Files ולהגדיר את הנתיב שלו ב־`LOCAL_DB`. [מקורות ומילון שדות](https://github.com/romimana/Planetary-analogs/blob/main/ML.md).

In [ ]:
import hashlib
import json
import sqlite3
from contextlib import closing
from pathlib import Path
from urllib.request import urlopen

import pandas as pd
from IPython.display import display

DATA_REF = "main"  # Set a commit SHA to reproduce a fixed release.
LOCAL_DB = None  # Example: "/content/ml-enriched-local.sqlite"

if LOCAL_DB:
    db_path = Path(LOCAL_DB).resolve()
else:
    base = f"https://raw.githubusercontent.com/romimana/Planetary-analogs/{DATA_REF}/data"
    with urlopen(f"{base}/ml-enriched.json", timeout=60) as response:
        expected_hash = json.load(response)["sha256"]
    with urlopen(f"{base}/ml-enriched.sqlite", timeout=120) as response:
        content = response.read()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise ValueError("Checksum mismatch. Re-run this cell or use a pinned DATA_REF.")
    db_path = Path("ml-enriched.sqlite").resolve()
    db_path.write_bytes(content)

with closing(sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)) as db:
    objects = db.execute(
        "SELECT name, type FROM sqlite_schema WHERE type IN ('table', 'view') "
        "AND name NOT LIKE 'sqlite_%' ORDER BY type, name"
    ).fetchall()
    tables = {
        name: pd.read_sql_query('SELECT * FROM "' + name.replace('"', '""') + '"', db)
        for name, _ in objects
    }

inventory = pd.DataFrame([
    {"name": name, "type": kind, "rows": len(tables[name]), "columns": tables[name].shape[1]}
    for name, kind in objects
])
display(inventory)

## תצוגה וחוסרים
שנו את `TABLE` ורוצו שוב. שמות הטבלאות מופיעים בתא הקודם. אפשר לגשת לכל טבלה גם דרך `tables["name"]`.

In [ ]:
TABLE = "earth_model_profiles"
df = tables[TABLE]
display(df.head(8))
missing = df.isna().sum()
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": missing,
    "missing_pct": (missing / max(len(df), 1) * 100).round(1),
}).sort_values("missing", ascending=False))

## שלוש טבלאות להשוואה
X_earth, X_mars ו־X_moon מכילות את כל ששת רכיבי הכימיה, את ששת מדדי השטח, את הקואורדינטות ואת יחידת המסלע ומקורה. מזהה הדגימה הוא האינדקס. ערך חסר נשאר חסר. כדי לראות תרחישי שמש זמינים הציגו את available_solar_times.
המשתנים עם הסיומת _ready מכילים רק את השדות שנבחרו לדירוג ורק דגימות שיש להן את כולם. ברירת המחדל של הדירוג משתמשת בארבעה רכיבי כימיה ובששת מדדי השטח. שלוש המטריצות מוכנות באותה סקאלה שמחושבת מכדור הארץ.

In [ ]:
import numpy as np

INCLUDE_SILICA = False
INCLUDE_MANGANESE = False
REQUIRE_MAPPED_GEOLOGY = False
# Leave both None for static matching. To include the Sun, pick one listed UTC per body.
EARTH_SOLAR_UTC = None
MARS_SOLAR_UTC = None
MOON_SOLAR_UTC = None

CHEMISTRY = {
    "tio2_wt_pct": "Titanium dioxide",
    "mgo_wt_pct": "Magnesium oxide",
    "cao_wt_pct": "Calcium oxide",
    "k2o_wt_pct": "Potassium oxide",
    "sio2_wt_pct": "Silica",
    "mno_wt_pct": "Manganese oxide",
}
TERRAIN = [
    "slope_median_deg", "slope_p90_deg", "slope_over_10_deg_fraction",
    "relief_p90_p10_m", "local_roughness_rms_m", "largest_gentle_patch_fraction",
]
# Edit this list to use a specific subset of terrain measures.
terrain_features = TERRAIN.copy()
exclude = ({"sio2_wt_pct"} if not INCLUDE_SILICA else set()) | (
    {"mno_wt_pct"} if not INCLUDE_MANGANESE else set()
)
chem_features = [name for name in CHEMISTRY if name not in exclude]
features = [*chem_features, *terrain_features]
profiles = {
    body: tables[f"{body}_model_profiles"].rename(columns={"longitude_e": "longitude"}).copy()
    for body in ("earth", "mars", "moon")
}
solar = tables["solar_scenarios"]
available_solar_times = solar[["body", "utc"]].drop_duplicates().reset_index(drop=True)

def join_solar(profile, body, key, utc):
    scenario = solar.loc[(solar["body"] == body) & (solar["utc"] == utc),
                         ["location_id", "horizontal_projection"]]
    if scenario.empty:
        raise ValueError(f"No published {body} solar scenario for {utc}")
    return profile.merge(scenario, left_on=key, right_on="location_id", how="left", validate="many_to_one")

solar_times = {"earth": EARTH_SOLAR_UTC, "mars": MARS_SOLAR_UTC, "moon": MOON_SOLAR_UTC}
if any(utc is not None for utc in solar_times.values()):
    if any(utc is None for utc in solar_times.values()):
        raise ValueError("Select one published solar scenario for each of the three bodies")
    for body, utc in solar_times.items():
        key = "site_id" if body == "earth" else "id"
        profiles[body] = join_solar(profiles[body], body, key, utc)
    features.append("horizontal_projection")

required = features + (["geology_unit"] if REQUIRE_MAPPED_GEOLOGY else [])
model_profiles = {
    body: frame.dropna(subset=required).set_index("id", verify_integrity=True)
    for body, frame in profiles.items()
}
display_columns = [
    *CHEMISTRY, *TERRAIN, "latitude", "longitude", "material",
    "geology_unit", "geology_description", "geology_product", "geology_url", "geology_locator",
]
if "horizontal_projection" in features:
    display_columns.append("horizontal_projection")
X_by_body = {
    body: profile.set_index("id", verify_integrity=True)[display_columns].copy()
    for body, profile in profiles.items()
}
X_earth, X_mars, X_moon = (X_by_body[body] for body in ("earth", "mars", "moon"))
selected = {body: profile[features].copy() for body, profile in model_profiles.items()}

for body, x in selected.items():
    if x.empty:
        raise ValueError(f"No eligible {body} rows for these settings")
    if not np.isfinite(x.to_numpy(dtype=float)).all() or x[chem_features].le(0).any().any():
        raise ValueError(f"Invalid value in selected {body} features")

def centered_log_ratios(frame):
    transformed = frame.copy()
    logs = np.log(frame[chem_features])
    transformed[chem_features] = logs.sub(logs.mean(axis=1), axis=0)
    return transformed.rename(columns={key: f"clr_{key}" for key in chem_features})

earth_vectors = centered_log_ratios(selected["earth"])
center = earth_vectors.median()
scale = earth_vectors.quantile(.75) - earth_vectors.quantile(.25)
# Keep zero-IQR features on their native transformed units so weights remain explicit.
scale = scale.mask(scale <= 1e-12, 1.0)
X_earth_ready = earth_vectors.sub(center).div(scale)
X_mars_ready = centered_log_ratios(selected["mars"]).sub(center).div(scale)
X_moon_ready = centered_log_ratios(selected["moon"]).sub(center).div(scale)
feature_scaling = pd.DataFrame({"earth_median": center, "divisor": scale})

with pd.option_context("display.max_columns", None):
    for body, frame in X_by_body.items():
        print(f"X_{body}: {len(frame):,} rows, {frame.shape[1]} columns; {len(selected[body]):,} eligible for ranking")
        display(frame.head(8))


## עיון בכל השורות
שנו את BODY_TO_VIEW ואת ROW_START כדי לראות חלק אחר של הטבלה. אפשר לערוך את הסינון בפייתון בלי לשנות את נתוני המקור.

In [ ]:
BODY_TO_VIEW = "earth"  # "earth", "mars", or "moon"
ROW_START = 0
PAGE_SIZE = 25

table = X_by_body[BODY_TO_VIEW]
print(f"Showing rows {ROW_START + 1}–{min(ROW_START + PAGE_SIZE, len(table))} of {len(table):,}")
with pd.option_context("display.max_columns", None, "display.max_rows", PAGE_SIZE):
    display(table.iloc[ROW_START:ROW_START + PAGE_SIZE])

## תרחישים סביבתיים
טבלאות `complete_earth`, `complete_mars`, `complete_moon` הן נקודת הפתיחה כשצריך תרחיש מלא. הטמפרטורות של MODIS, REMS ו־Diviner תלויות בשיטת המדידה. עיינו בשם השדה ובתיעוד לפני השוואה. טבלאות המקור משמרות מדידות אוויר ופני שטח עם חותמות זמן נפרדות. שמרו תרחישים מאותו אתר יחד בזמן הערכה.

In [ ]:
earth_scenarios = tables["complete_earth"]
BODY = "mars"  # Choose a body for the archive preview only.
target_scenarios = tables[f"complete_{BODY}"]
display(earth_scenarios[["site_id", "date", "pass", "air_temperature_k",
                         "temperature_k", "pressure_pa", "weather_distance_m"]].head())
temperature = "ground_brightness_k" if BODY == "mars" else "lunar_surface_bolometric_k"
display(target_scenarios[["id", "window_label", "local_solar_hour", "distance_m", temperature]].head())

## כיסוי כל השכבות לפי דגימה
`point_profiles["earth"]`, `point_profiles["mars"]`, `point_profiles["moon"]` מציגות את כל הדגימות הזמינות למידול. עמודות שמסתיימות ב־`_n` סופרות תצפיות. הערכים עצמם נמצאים ב־`earth_weather`, ב־`tables["earth_modis"]` וב־`tables["planetary_profiles"]`. השארנו את סדרות הזמן נפרדות כדי למנוע צירוף בין שעות שונות.
זווית השמש נשמרת ב־`solar_scenarios` לכל שלושת העולמות. חברו לפי `body` ולפי `location_id` (מזהה אתר בכדור הארץ, מזהה דגימה במאדים ובירח). תרחישי השמש מוגדרים מראש ומתוארכים. [מילון השדות והמקורות](https://github.com/romimana/Planetary-analogs/blob/main/DATA_CODEBOOK.md)

In [ ]:
earth_weather = tables["earth_environment"]
solar = tables["solar_scenarios"]

# Keep full scenarios separate. These per-sample profiles show availability without multiplying rows.
point_profiles = {
    "earth": tables["earth_model_profiles"].copy(),
    "mars": tables["mars_model_profiles"].copy(),
    "moon": tables["moon_model_profiles"].copy(),
}

def add_counts(profile, key, observations, observation_key, fields):
    grouped = observations.groupby(observation_key)
    for field in fields:
        profile[field + "_n"] = profile[key].map(grouped[field].count()).fillna(0).astype(int)

earth = point_profiles["earth"]
add_counts(earth, "site_id", earth_weather, "site_id",
           ["air_temperature_k", "pressure_pa", "wind_speed_10m_m_s", "soil_water_0_to_7cm_m3_m3"])
add_counts(earth, "site_id", tables["earth_modis"], "site_id", ["temperature_k"])
earth.rename(columns={"temperature_k_n": "surface_temperature_n"}, inplace=True)
add_counts(earth, "site_id", tables["earth_radar"], "site_id", ["median_dn"])

for body in ("mars", "moon"):
    station = tables["planetary_profiles"].loc[
        lambda df: (df["body"] == body) & (df["comparison_status"] == "rover_or_station")
    ]
    surface = "ground_brightness_k" if body == "mars" else "lunar_surface_bolometric_k"
    fields = [surface, "local_solar_hour"]
    if body == "mars":
        fields += ["air_temperature_k", "pressure_pa"]
    add_counts(point_profiles[body], "id", station, "id", fields)
    point_profiles[body].rename(columns={surface + "_n": "surface_temperature_n"}, inplace=True)

for body, profile in point_profiles.items():
    key = "site_id" if body == "earth" else "id"
    add_counts(profile, key, solar[solar["body"] == body], "location_id", ["solar_zenith_deg"])

add_counts(earth, "id", tables["earth_grain_notes"], "sample_id", ["source_grain_description_ja"])
add_counts(point_profiles["mars"], "id", tables["mars_texture_context"], "sample_id", ["texture_context"])
exact_grain = tables["moon_grain_bins"].query("join_level == 'same_aliquot'")
add_counts(point_profiles["moon"], "id", exact_grain, "sample_id", ["weight_pct"])
mars_wind = tables["mars_environment"].query("comparison_status == 'rover_or_station'")
add_counts(point_profiles["mars"], "id", mars_wind, "id", ["wind_speed_m_s"])

audit = pd.DataFrame({
    body: pd.Series({
        column: int((frame[column] == 0).sum() if column.endswith("_n") else frame[column].isna().sum())
        for column in frame
    }, dtype="Int64")
    for body, frame in point_profiles.items()
})
display(audit.loc[[*CHEMISTRY, *TERRAIN, "geology_unit", "surface_temperature_n",
                   "air_temperature_k_n", "pressure_pa_n", "solar_zenith_deg_n",
                   "local_solar_hour_n", "median_dn_n",
                   "wind_speed_10m_m_s_n", "soil_water_0_to_7cm_m3_m3_n", "wind_speed_m_s_n",
                   "source_grain_description_ja_n", "texture_context_n", "weight_pct_n"]])
display(point_profiles[BODY].head())
display(earth_weather.head())